# ارائه پروژه SimSiam روی CIFAR-10
این notebook همه مراحل پروژه را برای ارائه نشان می‌دهد.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT))
from utils.seed import seed_everything
seed_everything(42)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)

## معماری و تست dummy batch
دو view از یک تصویر وارد encoder، projector و predictor می‌شوند. stop-gradient در loss اعمال می‌شود.

In [ ]:
from models.simsiam import SimSiam
model=SimSiam().to(DEVICE)
x1=torch.rand(4,3,32,32).to(DEVICE)
x2=torch.rand(4,3,32,32).to(DEVICE)
loss=model(x1,x2)
print('Input:',tuple(x1.shape),'Loss:',round(loss.item(),4))
assert loss.ndim==0
print('Architecture test: PASS')

## CIFAR-10
برای ارائه از subset کوچک استفاده می‌کنیم؛ برای آزمایش کامل LIMIT را None کنید.

In [ ]:
from datasets.cifar10 import CIFAR10,CLASSES
LIMIT=256
try:
    train_set=CIFAR10('data',train=True,download=True,two_views=True,limit=LIMIT)
    print('Samples:',len(train_set),'Classes:',CLASSES)
except Exception as error:
    print('Dataset error:',error)

## Self-supervised pretraining
در این مرحله label برای loss استفاده نمی‌شود.

In [ ]:
from training.pretrain import train as pretrain
try:
    pretrained_model,history=pretrain(epochs=1,batch_size=16,limit=LIMIT,device=DEVICE,checkpoint='checkpoints/notebook_simsiam.pt')
    print(history)
except Exception as error:
    history=[]; print('Pretraining error:',error)

## Linear evaluation و supervised baseline

In [ ]:
from training.linear_eval import train as linear_train
try:
    classifier=linear_train(epochs=1,batch_size=16,limit=LIMIT,device=DEVICE,encoder_checkpoint='checkpoints/notebook_simsiam.pt',checkpoint='checkpoints/notebook_linear.pt')
    print('Linear evaluation: PASS')
except Exception as error: print('Linear evaluation error:',error)

In [ ]:
from training.supervised import train as supervised_train
try:
    baseline=supervised_train(epochs=1,batch_size=16,limit=LIMIT,device=DEVICE,checkpoint='checkpoints/notebook_supervised.pt')
    print('Supervised baseline: PASS')
except Exception as error: print('Supervised error:',error)

## تست checkpoint و نمایش loss

In [ ]:
from utils.checkpoint import load_checkpoint
try: load_checkpoint('checkpoints/missing.pt',SimSiam())
except FileNotFoundError as error: print('Expected error:',error)
if history:
    plt.plot(history,marker='o'); plt.title('SimSiam loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(); plt.show()

## اجرای دمو
پس از اجرای آموزش، سلول زیر را فعال کنید. دمو تصویر upload‌شده، کلاس پیش‌بینی‌شده و احتمال همه کلاس‌ها را نشان می‌دهد.

In [ ]:
# from app.demo import launch
# launch('checkpoints/notebook_linear.pt')

### جمع‌بندی
SimSiam بدون label representation یاد می‌گیرد؛ stop-gradient و negative cosine loss اجزای کلیدی هستند. کل pipeline روی CPU قابل اجراست.